# Assignment 7 | PCA Encoders (Linear, Nonlinear, & Deep)

In [ ]:
%pip install pandas
import numpy as np 
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler




covtype = pd.read_csv("/Users/blove/PycharmProjects/deepLearning/Data/covtype.csv")




## 0.1 | Load & Split Data

In [ ]:

X = covtype.drop(columns=["Cover_Type"])
y = covtype["Cover_Type"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)


continious_cols = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]


## 0.2 | Scale

In [ ]:
# Standardizing continious

from sklearn.preprocessing import StandardScaler

sc = StandardScaler().fit(X_train[continious_cols])
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continious_cols] = sc.transform(X_train[continious_cols])
X_val_scaled[continious_cols] = sc.transform(X_val[continious_cols])
X_test_scaled[continious_cols] = sc.transform(X_test[continious_cols])

## 0.3 | Baseline Classifier (no learned features)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


# baseline random forest on original features
rf = RandomForestClassifier(
n_estimators=200,
random_state=42,
n_jobs=-1
)

rf.fit(X_train, y_train)



# evaluate (val for tuning if needed, test for final baseline)
y_val_pred = rf.predict(X_val)
y_test_pred = rf.predict(X_test)

val_acc = accuracy_score(y_val, y_val_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")
test_acc = accuracy_score(y_test, y_test_pred)
test_macro_f1= f1_score(y_test, y_test_pred, average="macro")

print(f"Validation accuracy: {val_acc:.3f}, Macro-F1: {val_macro_f1:.3f}")
print(f"Test Accuracy: {test_acc:.3F}, macro-F1: {test_macro_f1:3F}")

## Part A | Linear autoencoder

In [ ]:
# Archtecture
# Input dimensionally
D = X_train.shape[1]

encoder_ln = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='linear', use_bias=False)
])

decoder_ln = tf.keras.Sequential([
    tf.keras.layers.Dense(D, activation='linear', use_bias=False)
])

autoencoder_ln = tf.keras.Sequential([encoder_ln, decoder_ln]) # Full AE = encoder followed by decoder

autoencoder_ln.compile(
    loss="mse", 
    optimizer="adam"
    )

In [ ]:
# Function for Parts A, B, & C

def clf_trainer(autoencoder, encoder):
    
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
        )   

    history_ae = autoencoder.fit(
        X_train_scaled,
        X_train_scaled,
        validation_data=(X_val_scaled, X_val_scaled),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop],
        verbose = 1 
        )

    Z_train = encoder.predict(X_train_scaled)
    Z_val = encoder.predict(X_val_scaled)
    Z_test = encoder.predict(X_test_scaled)
    
    num_classes = len(y_train.unique())

    clf = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(10,)),      # 10 features from the AE
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    clf.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history_clf = clf.fit(
    Z_train, y_train,
    validation_data=(Z_val, y_val),
    epochs=50,
    batch_size=256,
    verbose=1
    ) 

    y_pred = clf.predict(Z_test).argmax(axis=1)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    return {
    "history_ae": history_ae,
    "history_clf": history_clf,
    "classifier": clf,
    "Z_train": Z_train,
    "Z_val": Z_val,
    "Z_test": Z_test,
    "macro_f1": macro_f1
    }



In [ ]:
clf_trainer(autoencoder_ln, encoder_ln)

linear_results = clf_trainer(autoencoder_ln, encoder_ln)


## Part B | Nonlinear Autoencoder

In [ ]:
# Architecture


encoder_nln = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='tanh', use_bias=True)
])

decoder_nln = tf.keras.Sequential([
    tf.keras.layers.Dense(D, activation='linear', use_bias=False)
])

autoencoder_nln = tf.keras.Sequential([encoder_ln, decoder_ln]) 

autoencoder_nln.compile(
    loss="mse", 
    optimizer="adam"
    )


clf_trainer(autoencoder_nln, encoder_nln)

nonlinear_results = clf_trainer(autoencoder_nln, encoder_nln)


## Part C | Deep Autoencoder

In [ ]:
# Architecture


encoder_deep = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='tanh', use_bias=True)
])

decoder_deep = tf.keras.Sequential([
    tf.keras.layers.Dense(D, activation='linear', use_bias=False)
])

autoencoder_deep = tf.keras.Sequential([encoder_ln, decoder_ln]) 

autoencoder_deep.compile(
    loss="mse", 
    optimizer="adam"
    )


clf_trainer(autoencoder_deep, encoder_deep)
deep_results = clf_trainer(autoencoder_deep, encoder_deep)

# Analysis/Questions

## Graphs

In [ ]:
import matplotlib.pyplot as plt

all_results = {
    "Linear AE": linear_results,
    "Nonlinear AE": nonlinear_results,
    "Deep AE": deep_results
}

# Metrics for AE and Classifier
ae_metrics = ["loss", "val_loss"]
clf_metrics = ["accuracy", "val_accuracy"]

def plot_all_models(results_dict):
    plt.figure(figsize=(10, 5))
    for metric in ae_metrics: 
        for name, results in results_dict.items():                                                 
            hist = results["history_ae"]           
            plt.plot(hist.history[metric], label=name) #plots metric

        plt.title(f"Autoencoder {metric.replace('_', ' ').title()}")
        plt.xlabel("Epoch")
        plt.ylabel(metric.replace('_', ' ').title())
        plt.legend()
        plt.grid(True)
        plt.show()


    for metric in clf_metrics:
        for name, results in results_dict.items():
            hist = results["history_clf"]
            plt.plot(hist.history[metric], label=name)

        plt.title(f"Classifier {metric.replace('_', ' ').title()}")
        plt.xlabel("Epoch")
        plt.ylabel(metric.replace('_', ' ').title())
        plt.legend()
        plt.grid(True)
        plt.show()

plot_all_models(all_results)

## Final Table

In [ ]:

rows = []

for name, res in all_results.items():
    hist_clf = res["history_clf"]
    
    # Final validation accuracy
    val_acc = hist_clf.history["val_accuracy"][-1]
    
    # Final training accuracy
    train_acc = hist_clf.history["accuracy"][-1]
    
    # Test accuracy
    y_pred = res["classifier"].predict(res["Z_test"]).argmax(axis=1)
    test_acc = (y_pred == y_test).mean()
    
    # Macro-F1
    macro_f1 = res.get("macro_f1", None)
    
    rows.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Val Accuracy": val_acc,
        "Test Accuracy": test_acc,
        "Macro-F1": macro_f1
    })

df_results = pd.DataFrame(rows)
df_results

## Written Analysis

- The baseline classifier gives the best performance since it uses all 54 features.
- PCA & the Linear autoencoder lose information because they use a linear subspace.
- Nonlinear and deep autoencoder's learn nonlinear manifolds, which make them "better," for the purposes of this assignment.
- The macro-F1 confirms improvement amongst all classes, not just the majority ones. 